# 25장 입력값 표, 관찰 기간의 결과, 재정영향 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 25장의 공통 예시(진행성 신세포암, 신약 A 대 표준요법 B)로 입력값 표와 결과표, 분석기간을 시험의 관찰 기간으로 줄인 결과, 연차별 1인당 비용과 5년 재정영향을 R로 계산합니다. 자료 파일은 없고 모형의 입력값을 코드에 적습니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 추가 패키지는 쓰지 않습니다.


## a. 입력값 표와 결과표

### 셀 1. 분석기간을 인자로 둔 공통 예시 모형

In [ ]:
p <- list(pfs_med = 10, pfs_gam = 0.95, os_med = 28, os_gam = 1.15,   # 표준요법 B의 곡선(중앙값 개월, 모양)
          hr_pfs = 0.65, hr_os = 0.75,                                  # 신약 A의 위험비
          u_pf = 0.78, u_pd = 0.62, du_ae_A = 0.012, du_ae_B = 0.008,   # 효용과 이상반응의 QALY 손실
          c_drug_A = 190, c_drug_B = 120, c_pf = 40, c_pd = 250,        # 월 비용(만원)
          c_death = 800, c_ae_A = 120, c_ae_B = 80, disc = 0.045)       # 1회 비용(만원), 연 할인율
model <- function(p, horizon = 240, disc = TRUE) {    # 분석기간(개월)과 할인 여부를 인자로
  t <- 0:horizon
  d <- if (disc) 1 / (1 + p$disc)^((head(t, -1) + 0.5) / 12) else 1   # 각 달의 한가운데에서 할인
  sapply(c(A = "A", B = "B"), function(arm) {
    hp <- if (arm == "A") p$hr_pfs else 1; ho <- if (arm == "A") p$hr_os else 1
    s_os  <- exp(-log(2) * ho * (t / p$os_med)^p$os_gam)
    s_pfs <- pmin(exp(-log(2) * hp * (t / p$pfs_med)^p$pfs_gam), s_os)
    pf <- (head(s_pfs, -1) + tail(s_pfs, -1)) / 2                     # 반주기 보정
    pd <- (head(s_os - s_pfs, -1) + tail(s_os - s_pfs, -1)) / 2
    died <- head(s_os, -1) - tail(s_os, -1)
    c_drug <- p[[paste0("c_drug_", arm)]]
    c(cost = sum((pf * (c_drug + p$c_pf) + pd * p$c_pd + died * p$c_death) * d) + p[[paste0("c_ae_", arm)]],
      qaly = sum((pf * p$u_pf + pd * p$u_pd) / 12 * d) - p[[paste0("du_ae_", arm)]],
      drug = sum(pf * c_drug * d),                    # 1차 약값
      pf_months = sum(pf), pd_months = sum(pd))       # 상태에 머문 기간(개월, 할인하지 않음)
  })
}
r <- model(p)
round(r, 3)

### 셀 2. 입력값 표와 결과표를 모형의 값에서 만들기

In [ ]:
src <- c(pfs_med = "Pivotal trial, arm B", os_med = "Pivotal trial, arm B", hr_pfs = "Pivotal trial",
         hr_os = "Pivotal trial", u_pf = "Trial EQ-5D-5L, Korean value set", u_pd = "Trial EQ-5D-5L, Korean value set",
         c_drug_A = "List price", c_pd = "Claims data", disc = "HIRA guideline")
table1 <- data.frame(base_case = unlist(p[names(src)]), source = src)   # 사전의 값을 그대로 표로
table1
res <- data.frame(cost = r["cost", ], qaly = r["qaly", ])              # 결과표: 반올림하지 않은 값
res["A - B", ] <- res["A", ] - res["B", ]
res
c(icer = res["A - B", "cost"] / res["A - B", "qaly"],                  # 반올림 전 값으로 나눔
  from_table = round(res["A - B", "cost"]) / round(res["A - B", "qaly"], 3))   # 표에 적힌 2,872 / 0.511

### 셀 3. 반올림 규칙 확인하기

In [ ]:
round(c(2.5, 3.5, 0.5))              # 정확히 반인 값은 짝수 쪽으로
round(56.25, 1); round(0.125, 2)
sprintf("%.1f", 56.25)               # 글자로 바꿀 때도 같음
half_up <- function(x, digits = 0) floor(x * 10^digits + 0.5) / 10^digits   # 엑셀 ROUND처럼 올림
half_up(c(2.5, 56.25, 0.125), c(0, 1, 2))

## b. 관찰 기간의 결과

### 셀 4. 분석기간 20년과 30개월의 증분 결과

In [ ]:
inc <- function(r) c(d_cost = r["cost", "A"] - r["cost", "B"], d_qaly = r["qaly", "A"] - r["qaly", "B"],
                     icer = (r["cost", "A"] - r["cost", "B"]) / (r["qaly", "A"] - r["qaly", "B"]))
r30 <- model(p, horizon = 30)                         # 시험이 관찰한 30개월에서 끊음
tab <- rbind(years20 = inc(r), months30 = inc(r30))
round(tab, 3)
round(1 - tab["months30", 1:2] / tab["years20", 1:2], 3)   # 30개월 뒤에서 생긴 비율
round(r30[c("pf_months", "pd_months"), ], 1)               # 30개월 안에 각 상태에 머문 평균 기간

## c. 재정영향

### 셀 5. 연차별 1인당 비용

In [ ]:
yearly <- function(p, what = "cost") {               # 치료 시작 후 1-5년차의 1인당 값(할인하지 않음)
  cum <- sapply(1:5, function(k) model(p, horizon = 12 * k, disc = FALSE)[what, ])   # 2행(A, B) x 5열
  out <- cum - cbind(0, cum[, -5])                    # k년 누적 - (k-1)년 누적
  colnames(out) <- paste0("year", 1:5); out
}
cy <- yearly(p)
round(rbind(cy, diff = cy["A", ] - cy["B", ]), 1)

### 셀 6. 5년 재정영향 (표 25-8)

In [ ]:
bia <- function(cA, cB, n = 600, share = c(0.1, 0.2, 0.3, 0.4, 0.5)) {
  start_A <- n * share                                # 급여 후 해마다 신약 A로 시작하는 환자 수
  out <- matrix(0, 3, 5, dimnames = list(c("without_A", "with_A", "net"), paste0("year", 1:5)))
  for (y in 1:5) for (s in 1:y) {                     # 급여 y년차에 치료 중인 s년 코호트
    k <- y - s + 1                                    # 그 코호트의 치료 연차
    out["without_A", y] <- out["without_A", y] + n * cB[k]
    out["with_A", y]    <- out["with_A", y] + start_A[s] * cA[k] + (n - start_A[s]) * cB[k]
    out["net", y]       <- out["net", y] + start_A[s] * (cA[k] - cB[k])
  }
  cbind(out, total = rowSums(out)) / 10000            # 억 원
}
round(bia(cy["A", ], cy["B", ]), 1)
dy <- yearly(p, "drug")                               # 1차 약값만
round(bia(dy["A", ], 0 * dy["B", ])["net", ], 1)      # 신약 A 약품비 총액

### 셀 7. 가정을 바꾼 재정영향

In [ ]:
price <- uniroot(function(x) { r <- model(modifyList(p, list(c_drug_A = x)))
  inc(r)[["icer"]] - 5000 }, c(150, 190), tol = 1e-9)$root   # ICER가 5,000이 되는 월 약값
price
cy_p <- yearly(modifyList(p, list(c_drug_A = price)))
sc <- list(base        = bia(cy["A", ], cy["B", ]),
           share_half  = bia(cy["A", ], cy["B", ], share = c(0.05, 0.10, 0.15, 0.20, 0.25)),
           n_900       = bia(cy["A", ], cy["B", ], n = 900),
           price_5000  = bia(cy_p["A", ], cy_p["B", ]),
           drug_only   = bia(dy["A", ], dy["B", ]))
round(t(sapply(sc, function(b) b["net", c("year3", "total")])), 1)
round(cy_p["A", 1] - cy_p["B", 1], 1)                 # 그 가격에서 치료 1년차의 1인당 비용 차이

## d. 과제 정답

### 셀 8. 과제 1 정답. 환급률에 따른 ICER (표 25-7)

In [ ]:
refund <- c(0, 0.05, 1 - price / 190, 0.10)           # 환급률. 셋째는 ICER가 5,000이 되는 7.7%
real <- 190 * (1 - refund)                            # 실제로 내는 월 약값
icer_real <- sapply(real, function(x) inc(model(modifyList(p, list(c_drug_A = x))))[["icer"]])
data.frame(refund = round(refund, 3), real = round(real, 1), icer_list = round(inc(r)[["icer"]]), icer_real = round(icer_real))

### 셀 9. 과제 2 정답. 어디서 끊느냐에 따른 관찰 기간만의 결과

In [ ]:
r18 <- model(p, horizon = 18)                         # 중앙 추적 기간 18개월에서 끊으면
tab2 <- rbind(tab, months18 = inc(r18))
round(tab2, 3)
round(1 - sweep(tab2[c("months30", "months18"), 1:2], 2, tab2["years20", 1:2], "/"), 3)   # 끊은 뒤에서 생긴 비율